# Notebook 6 | Chain Rule of Probability

In [1]:
from foundations_of_probability_and_statistics.cars.car_distribution import create_car_distribution
from foundations_of_probability_and_statistics.cars.car_distribution import create_joint_car_distribution

import pandas as pd

# show all rows of data frames and series per default
pd.set_option("display.max_rows", None)

## The chain rule: factoring a joint one variable at a time

How is a joint of arbitrary size decomposed into tractable pieces? One variable is factored out at a time with the product rule (notebook 5), conditioning each new factor on all variables factored out before. For the cars, in full form first and then in shorthand,

$$p(B = b, H = h, C = c) = p(B = b) \, p(H = h \mid B = b) \, p(C = c \mid B = b, H = h),$$
$$p(b, h, c) = p(b) \, p(h \mid b) \, p(c \mid b, h).$$

The rule applies whenever variables admit a sequential ordering, e.g. successive production stages of a car. The central observation: the trailing term reduces to $p(c \mid b)$ by the conditional independence of notebook 4, so the chain rule identifies precisely where the model requires fewer parameters.

## Derivation

1. Factor out the last variable: $p(b, h, c) = p(b, h) \, p(c \mid b, h)$.
2. Factor out the next: $p(b, h) = p(b) \, p(h \mid b)$.
3. Substitute back: $p(b, h, c) = p(b) \, p(h \mid b) \, p(c \mid b, h)$.
4. Use the conditional independence of notebook 4: $p(c \mid b, h) = p(c \mid b)$, so the chain reduces to the product rule.

The standard error is to attribute significance to the ordering: every factorization order yields the same joint, and the generalization cell verifies this by reconstructing the table in reverse order.

## Worked example (by hand)

With $p(500 \mid \text{Ferrari}, \text{red}) = p(500 \mid \text{Ferrari})$ by conditional independence:

$$p(\text{Ferrari}, 500, \text{red}) = 0.2 \cdot 0.4 \cdot 0.6 = 0.048.$$

Read the three factors as a story: one car in five is a Ferrari; four in ten Ferraris make 500 hp; six in ten Ferraris are red, regardless of horsepower. Each factor answers a narrower question than the joint it builds.

In [2]:
import math

joint = create_joint_car_distribution()
car_distribution = create_car_distribution()

# chain rule with the conditional independence shortcut p(500 | Ferrari, red) = p(500 | Ferrari)
p_ferrari = car_distribution.brand_rv.pmf(2)
p_500_given_ferrari = car_distribution.horsepower_rvs[2].pmf(500)
p_red_given_ferrari = car_distribution.color_rvs[2].pmf(1)
assert math.isclose(p_ferrari * p_500_given_ferrari * p_red_given_ferrari, 0.2 * 0.4 * 0.6)
assert math.isclose(joint.loc[("Ferrari", 500, "red")], 0.048)
joint.loc[("Ferrari", 500, "red")]

np.float64(0.04800000000000001)

## Generalization

The other variable ordering $p(b, h, c) = p(b) \, p(c \mid b) \, p(h \mid b, c)$ gives the same table, again because horsepower and color are conditionally independent given the brand. Notebook 7 reverses the perspective: instead of building up, sum over a partition to break a marginal down.

In [3]:
# alternative ordering: p(b, h, c) = p(b) p(c | b) p(h | b, c) with p(h | b, c) = p(h | b)
rebuilt = {
    (car_distribution.brand_names[b], int(h), car_distribution.color_names[c]): car_distribution.brand_rv.pmf(b)
    * car_distribution.color_rvs[b].pmf(c)
    * car_distribution.horsepower_rvs[b].pmf(h)
    for b in car_distribution.horsepower_rvs
    for h in car_distribution.horsepower_rvs[b].xk
    for c in car_distribution.color_rvs[b].xk
}
assert all(math.isclose(joint.loc[key], value) for key, value in rebuilt.items())
assert math.isclose(sum(rebuilt.values()), 1.0)
len(rebuilt)

38

## References

- Blitzstein, J. K., Hwang, J. (2019): "Introduction to Probability", 2nd ed., Chapman & Hall/CRC, chapter "Conditional Probability", https://www.routledge.com/Introduction-to-Probability-Second-Edition/Blitzstein-Hwang/p/book/9781138369917 (free PDF: https://probabilitybook.net/).
- Wasserman, L. (2004): "All of Statistics: A Concise Course in Statistical Inference", Springer Texts in Statistics, chapter "Probability", https://doi.org/10.1007/978-0-387-21736-9.
- scipy.stats.rv_discrete, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_discrete.html.